In [1]:
import json
import re
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm import tqdm

banned_words = ['porn', 'sex', 'child', 'drugs', 'lolita']

def contains_banned_words(text, banned_words):
    text_lower = text.lower()
    return any(word in text_lower for word in banned_words)

def is_full_sentence(text):
    text = text.strip()
    return bool(re.match(r'^[A-Z].*[.!?]$', text))

def extract_valid_paragraphs(html_content):
    if not isinstance(html_content, str) or not html_content:
        return ""

    soup = BeautifulSoup(html_content, 'html.parser')
    paragraphs = soup.find_all('p')

    valid_paragraphs = []
    for paragraph in paragraphs:
        text = paragraph.get_text().strip()
        if not text:
            continue

        if contains_banned_words(text, banned_words):
            continue

        sentences = re.split(r'(?<=[.!?])\s+', text)
        full_sentences = [s for s in sentences if is_full_sentence(s)]

        if full_sentences:
            valid_paragraphs.append(' '.join(full_sentences))

    return ' '.join(valid_paragraphs)

In [2]:
def jsonl_full_to_enriched(input_jsonl, output_jsonl, html_key="html_code", raw_key="raw_content", limit_rows=None):
    input_jsonl = Path(input_jsonl)
    output_jsonl = Path(output_jsonl)
    output_jsonl.parent.mkdir(parents=True, exist_ok=True)

    written = 0
    bad_json = 0

    with input_jsonl.open("r", encoding="utf-8", errors="replace") as fin, \
         output_jsonl.open("w", encoding="utf-8") as fout:

        for line in tqdm(fin, desc="Processing JSONL → enriched"):
            if limit_rows is not None and written >= limit_rows:
                break

            line = line.strip()
            if not line:
                continue

            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                bad_json += 1
                continue

            html = rec.get(html_key, "")
            rec[raw_key] = extract_valid_paragraphs(html)

            # Keep ALL snapshots and keep html_code
            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
            written += 1

    print(f"Done. Wrote {written:,} records to: {output_jsonl.resolve()}")
    print(f"Skipped bad JSON lines: {bad_json:,}")
    return {"written": written, "bad_json": bad_json, "output": str(output_jsonl.resolve())}

In [3]:
stats = jsonl_full_to_enriched(
    input_jsonl="test_export.jsonl",
    output_jsonl="test_export_enriched.jsonl",
    limit_rows=None   # set to 1000 for a quick test
)
stats

Processing JSONL → enriched: 11403638it [17:29:27, 181.10it/s] 

Done. Wrote 11,403,638 records to: /home/darknet/2026-01-27_201419_domain_with_snapshots/test_export_enriched.jsonl
Skipped bad JSON lines: 0


{'written': 11403638,
 'bad_json': 0,
 'output': '/home/darknet/2026-01-27_201419_domain_with_snapshots/test_export_enriched.jsonl'}

In [5]:
from itertools import islice

with open("test_export_enriched.jsonl", "r", encoding="utf-8") as f:
    for line in islice(f, 100):
        rec = json.loads(line)
        print("keys:", list(rec.keys()))
        print("raw_content preview:", (rec.get("raw_content","")[:200] + "...") if rec.get("raw_content") else "")
        print("-"*80)

keys: ['created_at', 'darknet_type', 'domain_id', 'domain_url', 'file_size', 'html_file_location', 'page_id', 'path', 'path_hash', 'snapshot_hash', 'snapshot_id', 'tags', 'title', 'html_code', 'raw_content']
raw_content preview: 
--------------------------------------------------------------------------------
keys: ['created_at', 'darknet_type', 'domain_id', 'domain_url', 'file_size', 'html_file_location', 'page_id', 'path', 'path_hash', 'snapshot_hash', 'snapshot_id', 'tags', 'title', 'html_code', 'raw_content']
raw_content preview: You are not logged in....
--------------------------------------------------------------------------------
keys: ['created_at', 'darknet_type', 'domain_id', 'domain_url', 'file_size', 'html_file_location', 'page_id', 'path', 'path_hash', 'snapshot_hash', 'snapshot_id', 'tags', 'title', 'html_code', 'raw_content']
raw_content preview: You are not logged in. Hey all, I can't see any posts yet. How long before I have access? Or if someone coulld redirect me t